In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth
/kaggle/input/mask-rcnn-models/pytorch/default/10/MaskrcnnMobileNetV2_best.pt
/kaggle/input/mask-rcnn-models/pytorch/default/10/Maskrcnn_best.pt
/kaggle/input/resnet50/resnet50_weights_tf_dim_ordering_tf_kernels.h5
/kaggle/input/resnet50/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5
/kaggle/input/resnet50/imagenet_class_index.json
/kaggle/input/se_resnet50/pytorch/default/1/se_resnet50-ce0d4300.pth
/kaggle/input/csiro-biomass/sample_submission.csv
/kaggle/input/csiro-biomass/train.csv
/kaggle/input/csiro-biomass/test.csv
/kaggle/input/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/csiro-biomass/train/ID2099464826.jpg
/kaggle/input/csiro-biomass/train/ID2037861084.jpg
/kaggle/input/csiro-biomass/train/ID1211362607.jpg
/kaggle/input/csiro-biomass/train/ID1853508321.jpg
/kaggle/input/csiro-biomass/train/ID193102215.jpg
/kaggle/input/csiro-biomass/train/ID698608346.jpg
/kaggle/input/csir

In [2]:
# ============================================================================
# CELL 1: Setup and Data Loading
# ============================================================================
import numpy as np
import pandas as pd
import os
import torch
import torchvision
import torch.nn as nn
import pytorch_lightning as pl
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import v2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pytorch_lightning import Trainer
from tqdm import tqdm
import cv2

# Load data
train_file_path = "/kaggle/input/csiro-biomass/train.csv"
test_file_path = "/kaggle/input/csiro-biomass/test.csv"

train_pd = pd.read_csv(train_file_path)
test_pd_original = pd.read_csv(test_file_path)  # Keep original for submission
test_pd = test_pd_original.copy()  # Working copy for predictions

print(f"Train shape: {train_pd.shape}")
print(f"Test shape: {test_pd.shape}")

Train shape: (1785, 9)
Test shape: (5, 3)


In [3]:
# ============================================================================
# CELL 2: HEIGHT PREDICTION
# ============================================================================
print("\n=== STEP 1: Height Prediction ===")

height_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)
path = "/kaggle/input/csiro-biomass"
local_weights = "/kaggle/input/mask-rcnn-models/pytorch/default/10/Maskrcnn_best.pt"

try:
    state_dict = torch.load(local_weights, map_location="cpu", weights_only=False)
    height_model = state_dict
    height_model.eval()
    
    for index, image_path in test_pd["image_path"].items():
        image_path = os.path.join(path, image_path)
        image = Image.open(image_path).convert("RGB")
        
        image_transform = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        image_tensor = image_transform(image)
        
        with torch.no_grad():
            outputs = height_model([image_tensor])
        
        output_pixels = outputs[0]["boxes"].cpu().numpy()
        x1, y1, x2, y2 = output_pixels[0]
        image_in_cm = y2 / 2.54
        test_pd.loc[index, "Height_Ave_cm"] = image_in_cm
    
    print("✅ Height predictions completed using Mask R-CNN")
except:
    # Fallback to mean height
    mean_height = train_pd["Height_Ave_cm"].mean()
    test_pd["Height_Ave_cm"] = mean_height
    print(f"✅ Height predictions using mean: {mean_height:.2f}")


=== STEP 1: Height Prediction ===
✅ Height predictions completed using Mask R-CNN


species model

In [4]:
# ============================================================================
# CELL 3: SPECIES PREDICTION
# ============================================================================
print("\n=== STEP 2: Species Prediction ===")

# Global Encoders
SPECIES_LE = LabelEncoder()
TARGET_LE = LabelEncoder()

SPECIES_LE.fit(train_pd["Species"].astype(str).unique())
TARGET_LE.fit(train_pd["target_name"].astype(str).unique())

def safe_encode(le, val):
    """Encodes labels; returns 0 if label is unseen."""
    val_str = str(val)
    if val_str in le.classes_:
        return le.transform([val_str])[0]
    return 0

class SpeciesDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        tabular = torch.tensor([
            float(row["target_name"]), 
            float(row["Height_Ave_cm"])
        ], dtype=torch.float32)
        
        y = torch.tensor(int(row["Species"]), dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        return image, tabular, y

class SpeciesDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch_size=16, num_workers=2):
        super().__init__()
        self.train_df, self.valid_df = train_df.copy(), valid_df.copy()
        self.root_dir, self.batch_size, self.num_workers = root_dir, batch_size, num_workers

    def setup(self, stage=None):
        for df in [self.train_df, self.valid_df]:
            if not np.issubdtype(df["Species"].dtype, np.number):
                df["Species"] = df["Species"].apply(lambda x: safe_encode(SPECIES_LE, x))
            if not np.issubdtype(df["target_name"].dtype, np.number):
                df["target_name"] = df["target_name"].apply(lambda x: safe_encode(TARGET_LE, x))

        self.train_tf = v2.Compose([
            v2.RandomResizedCrop(224), v2.RandomHorizontalFlip(), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.valid_tf = v2.Compose([
            v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        self.train_ds = SpeciesDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = SpeciesDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

class EfficientNetSpeciesClassifier(pl.LightningModule):
    def __init__(self, num_classes, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.base_model = models.efficientnet_b0(weights=None)
        
        local_weights = "/kaggle/input/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth"
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            new_state_dict = {}
            for k, v in state_dict.items():
                n = k.replace("conv_stem", "features.0.0").replace("bn1", "features.0.1")
                if "blocks" in n:
                    p = n.split(".")
                    b_idx = int(p[1]) + 1
                    sub = ".".join(p[2:])
                    if b_idx == 1:
                        sub = sub.replace("conv_dw", "block.0.0").replace("bn1", "block.0.1")
                        sub = sub.replace("se.conv_reduce", "block.1.fc1").replace("se.conv_expand", "block.1.fc2")
                        sub = sub.replace("conv_pw", "block.2.0").replace("bn2", "block.2.1")
                    else:
                        sub = sub.replace("conv_pw", "block.0.0").replace("bn1", "block.0.1")
                        sub = sub.replace("conv_dw", "block.1.0").replace("bn2", "block.1.1")
                        sub = sub.replace("se.conv_reduce", "block.2.fc1").replace("se.conv_expand", "block.2.fc2")
                        sub = sub.replace("conv_pwl", "block.3.0").replace("bn3", "block.3.1")
                    n = f"features.{b_idx}.{sub}"
                n = n.replace("conv_head", "features.8.0").replace("bn2", "features.8.1")
                new_state_dict[n] = v
            self.base_model.load_state_dict(new_state_dict, strict=False)

        self.img_dim = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Identity()
        self.tabular_net = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 32))
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 32, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, img, tabular):
        return self.head(torch.cat([self.base_model(img), self.tabular_net(tabular)], dim=1))

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        loss = self.loss_fn(self(img, tab), y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        loss = self.loss_fn(self(img, tab), y)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# Training
image_root_dir = "/kaggle/input/csiro-biomass"
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42, stratify=train_pd["Species"])

datamodule = SpeciesDataModule(train_df=train_df, valid_df=valid_df, root_dir=image_root_dir)
datamodule.setup()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
species_model = EfficientNetSpeciesClassifier(num_classes=len(SPECIES_LE.classes_)).to(device)

trainer = Trainer(accelerator="gpu", devices=1, max_epochs=10)
trainer.fit(species_model, datamodule)

# Inference
species_model.eval()
species_model.to(device)

inf_tf = v2.Compose([
    v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

species_results = []
with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting Species"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            img_tensor = inf_tf(img).unsqueeze(0).to(device)
            
            t_idx = safe_encode(TARGET_LE, row["target_name"])
            tab_tensor = torch.tensor([[t_idx, float(row["Height_Ave_cm"])]], dtype=torch.float32).to(device)
            
            logits = species_model(img_tensor, tab_tensor)
            class_idx = logits.argmax(dim=1).item()
            actual_name = SPECIES_LE.inverse_transform([class_idx])[0]
            species_results.append(actual_name)
        except:
            species_results.append(SPECIES_LE.classes_[0])

test_pd["Species"] = species_results
print("✅ Species predictions completed")


=== STEP 2: Species Prediction ===


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
2026-01-15 16:14:17.778773: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768493658.106656      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768493658.199285      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768493659.013193      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same 

┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_model  │ EfficientNet     │  4.0 M │ train │     0 │
│ 1 │ tabular_net │ Sequential       │  2.3 K │ train │     0 │
│ 2 │ head        │ Sequential       │  177 K │ train │     0 │
│ 3 │ loss_fn     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.2 M                                                                                                
Total estimated model params size (MB): 16                                                                         
Modules in train mode: 347                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=10` reached.


Predicting Species: 100%|██████████| 5/5 [00:00<00:00, 10.88it/s]

✅ Species predictions completed


In [5]:
# ============================================================================
# CELL 4: STATE PREDICTION
# ============================================================================
print("\n=== STEP 3: State Prediction ===")

STATE_LE = LabelEncoder()
STATE_LE.fit(train_pd["State"].astype(str).unique())

class StateDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        tabular = torch.tensor([
            float(row["target_name"]), 
            float(row["Height_Ave_cm"]), 
            float(row["Species"])
        ], dtype=torch.float32)

        y = torch.tensor(row["State"], dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        return image, tabular, y

class StateDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch_size=16, num_workers=2):
        super().__init__()
        self.train_df, self.valid_df = train_df.copy(), valid_df.copy()
        self.root_dir, self.batch_size, self.num_workers = root_dir, batch_size, num_workers

    def setup(self, stage=None):
        for df in [self.train_df, self.valid_df]:
            if not np.issubdtype(df["State"].dtype, np.number):
                df["State"] = df["State"].apply(lambda x: safe_encode(STATE_LE, x))
            if not np.issubdtype(df["target_name"].dtype, np.number):
                df["target_name"] = df["target_name"].apply(lambda x: safe_encode(TARGET_LE, x))
            if not np.issubdtype(df["Species"].dtype, np.number):
                df["Species"] = df["Species"].apply(lambda x: safe_encode(SPECIES_LE, x))

        self.train_tf = v2.Compose([
            v2.RandomResizedCrop(224), v2.RandomHorizontalFlip(), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.valid_tf = v2.Compose([
            v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.train_ds = StateDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = StateDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

class EfficientNetStateClassifier(pl.LightningModule):
    def __init__(self, num_classes, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.base_model = models.efficientnet_b0(weights=None)
        
        local_weights = "/kaggle/input/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth"
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            new_state_dict = {}
            for k, v in state_dict.items():
                n = k.replace("conv_stem", "features.0.0").replace("bn1", "features.0.1")
                if "blocks" in n:
                    p = n.split(".")
                    b_idx = int(p[1]) + 1
                    sub = ".".join(p[2:])
                    if b_idx == 1:
                        sub = sub.replace("conv_dw", "block.0.0").replace("bn1", "block.0.1")
                        sub = sub.replace("se.conv_reduce", "block.1.fc1").replace("se.conv_expand", "block.1.fc2")
                        sub = sub.replace("conv_pw", "block.2.0").replace("bn2", "block.2.1")
                    else:
                        sub = sub.replace("conv_pw", "block.0.0").replace("bn1", "block.0.1")
                        sub = sub.replace("conv_dw", "block.1.0").replace("bn2", "block.1.1")
                        sub = sub.replace("se.conv_reduce", "block.2.fc1").replace("se.conv_expand", "block.2.fc2")
                        sub = sub.replace("conv_pwl", "block.3.0").replace("bn3", "block.3.1")
                    n = f"features.{b_idx}.{sub}"
                n = n.replace("conv_head", "features.8.0").replace("bn2", "features.8.1")
                new_state_dict[n] = v
            self.base_model.load_state_dict(new_state_dict, strict=False)

        self.img_dim = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Identity()
        self.tabular_net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 32))
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 32, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, img, tabular):
        return self.head(torch.cat([self.base_model(img), self.tabular_net(tabular)], dim=1))

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab)
        loss = self.loss_fn(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab)
        loss = self.loss_fn(logits, y)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# Training
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42, stratify=train_pd["State"])
datamodule = StateDataModule(train_df=train_df, valid_df=valid_df, root_dir=image_root_dir)
datamodule.setup()

state_model = EfficientNetStateClassifier(num_classes=len(STATE_LE.classes_)).to(device)
trainer = Trainer(accelerator="gpu", devices=1, max_epochs=10)
trainer.fit(state_model, datamodule)

# Inference
state_model.eval()
state_model.to(device)

state_results = []
with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting State"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            img_tensor = inf_tf(img).unsqueeze(0).to(device)
            
            t_idx = safe_encode(TARGET_LE, row["target_name"])
            s_idx = safe_encode(SPECIES_LE, row["Species"])
            h_val = float(row["Height_Ave_cm"])
            
            tab_tensor = torch.tensor([[t_idx, h_val, s_idx]], dtype=torch.float32).to(device)
            
            logits = state_model(img_tensor, tab_tensor)
            class_idx = logits.argmax(dim=1).item()
            actual_state = STATE_LE.inverse_transform([class_idx])[0]
            state_results.append(actual_state)
        except:
            state_results.append(STATE_LE.classes_[0])

test_pd["State"] = state_results
print("✅ State predictions completed")


=== STEP 3: State Prediction ===


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_model  │ EfficientNet     │  4.0 M │ train │     0 │
│ 1 │ tabular_net │ Sequential       │  2.3 K │ train │     0 │
│ 2 │ head        │ Sequential       │  176 K │ train │     0 │
│ 3 │ loss_fn     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.2 M                                                                                                
Total estimated model params size (MB): 16                                                                         
Modules in train mode: 348                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=10` reached.


Predicting State: 100%|██████████| 5/5 [00:00<00:00, 14.62it/s]

✅ State predictions completed


In [6]:
# ============================================================================
# CELL 5: NDVI PREDICTION
# ============================================================================
print("\n=== STEP 4: NDVI Prediction ===")

def image_to_mask(image_path):
    """Applies HSV masking to isolate green vegetation."""
    image = cv2.imread(image_path)
    if image is None:
        return np.zeros((224, 224, 3), dtype=np.uint8)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    return cv2.bitwise_and(image, image, mask=mask)

class PreGSSHDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["image_path"])
        masked_img = image_to_mask(img_path)
        
        s_idx = safe_encode(SPECIES_LE, row["Species"])
        t_idx = safe_encode(TARGET_LE, row["target_name"])
        ndvi_val = row["Pre_GSHH_NDVI"] if "Pre_GSHH_NDVI" in self.df.columns else 0.0
        
        tabular = torch.tensor([
            float(s_idx),
            float(ndvi_val),
            float(row["Height_Ave_cm"]),
            float(t_idx)
        ], dtype=torch.float32)

        target = torch.tensor([ndvi_val], dtype=torch.float32)

        if self.transform:
            image = self.transform(masked_img)
        else:
            image = torch.tensor(masked_img).permute(2, 0, 1).float()

        return image, tabular, target

class PreGSSHDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch=16):
        super().__init__()
        self.train_df, self.valid_df = train_df, valid_df
        self.root_dir, self.batch = root_dir, batch
        self.tfs = v2.Compose([
            v2.ToImage(), v2.Resize((224, 224)),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def setup(self, stage=None):
        self.train_ds = PreGSSHDataset(self.train_df, self.root_dir, self.tfs)
        self.valid_ds = PreGSSHDataset(self.valid_df, self.root_dir, self.tfs)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch, shuffle=True, num_workers=2)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch, num_workers=2)

class PreGSSHModel(pl.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.resnet = models.resnet50(weights=None)
        
        local_weights = '/kaggle/input/se_resnet50/pytorch/default/1/se_resnet50-ce0d4300.pth'
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            new_state_dict = {k.replace("layer0.", "").replace("last_linear", "fc"): v for k, v in state_dict.items()}
            self.resnet.load_state_dict(new_state_dict, strict=False)

        self.img_dim = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        self.tabular_net = nn.Sequential(
            nn.Linear(4, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32)
        )
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 32, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, img, tabular):
        return self.head(torch.cat([self.resnet(img), self.tabular_net(tabular)], dim=1))

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        loss = self.loss_fn(self(img, tab), y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        loss = self.loss_fn(self(img, tab), y)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# Training
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42)
datamodule = PreGSSHDataModule(train_df, valid_df, image_root_dir)

ndvi_model = PreGSSHModel()
trainer = Trainer(accelerator="gpu", devices=1, max_epochs=10)
trainer.fit(ndvi_model, datamodule)

# Inference
ndvi_model.eval()
ndvi_model.to(device)

inf_tfs = v2.Compose([
    v2.ToImage(), v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

ndvi_results = []
with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting NDVI"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            masked = image_to_mask(img_path)
            img_tensor = inf_tfs(masked).unsqueeze(0).to(device)
            
            s_idx = safe_encode(SPECIES_LE, row["Species"])
            t_idx = safe_encode(TARGET_LE, row["target_name"])
            
            tab_tensor = torch.tensor([[
                float(s_idx), 0.0, float(row["Height_Ave_cm"]), float(t_idx)
            ]], dtype=torch.float32).to(device)
            
            pred_ndvi = ndvi_model(img_tensor, tab_tensor).item()
            ndvi_results.append(pred_ndvi)
        except:
            ndvi_results.append(0.0)

test_pd["Pre_GSHH_NDVI"] = ndvi_results
print("✅ NDVI predictions completed")



=== STEP 4: NDVI Prediction ===


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ resnet      │ ResNet     │ 23.5 M │ train │     0 │
│ 1 │ tabular_net │ Sequential │  2.4 K │ train │     0 │
│ 2 │ head        │ Sequential │  266 K │ train │     0 │
│ 3 │ loss_fn     │ MSELoss    │      0 │ train │     0 │
└───┴─────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 23.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 23.8 M                                                                                               
Total estimated model params size (MB): 95                                                                         
Modules in train mode: 161                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=10` reached.


Predicting NDVI: 100%|██████████| 5/5 [00:00<00:00, 10.10it/s]

✅ NDVI predictions completed


In [7]:
# ============================================================================
# CELL 6: SPECIES-SPECIFIC BIOMASS PREDICTION
# ============================================================================
print("\n=== STEP 5: Final Biomass Prediction (Multi-Model Approach) ===")

# 1. Setup Encoders
TARGET_NAME_LE = LabelEncoder()
TARGET_NAME_LE.fit(train_pd["target_name"].astype(str).unique())

# 2. Generic Dataset & Model Classes
class BiomassDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(os.path.join(self.root_dir, row["image_path"])).convert("RGB")
        except:
            img = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform: img = self.transform(img)
        
        tab = torch.tensor([
            float(safe_encode(STATE_LE, row["State"])),
            float(safe_encode(SPECIES_LE, row["Species"])),
            float(row["Pre_GSHH_NDVI"]),
            float(row["Height_Ave_cm"]),
            float(safe_encode(TARGET_NAME_LE, row["target_name"]))
        ], dtype=torch.float32)

        if self.is_train:
            return img, tab, torch.tensor(row["target"], dtype=torch.float32)
        return img, tab

class BiomassLightningModel(pl.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.backbone = models.resnet50(weights=None)
        self.img_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.tab_net = nn.Sequential(nn.Linear(5, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 32))
        self.head = nn.Sequential(nn.Linear(self.img_dim + 32, 128), nn.ReLU(), nn.Linear(128, 1))
        self.loss_fn = nn.MSELoss()

    def forward(self, img, tab):
        return self.head(torch.cat([self.backbone(img), self.tab_net(tab)], dim=1)).squeeze(1)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        loss = self.loss_fn(self(img, tab), y)
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# 3. Training Loop per Species
unique_species = train_pd["Species"].unique()
trained_models = {} # Dictionary to store models for inference
biomass_tfs = v2.Compose([
    v2.Resize((224, 224)), v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

for sp in unique_species:
    print(f"\n--- Training Model for Species: {sp} ---")
    sp_train_data = train_pd[train_pd["Species"] == sp]
    
    # Check if there's enough data to split
    if len(sp_train_data) < 5:
        print(f"⚠️ Not enough data for {sp}, skipping training.")
        continue

    train_df, valid_df = train_test_split(sp_train_data, test_size=0.15, random_state=42)
    
    train_ds = BiomassDataset(train_df, image_root_dir, biomass_tfs)
    valid_ds = BiomassDataset(valid_df, image_root_dir, biomass_tfs)
    
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=8)
    
    model = BiomassLightningModel()
    trainer = Trainer(accelerator="gpu", devices=1, max_epochs=5, enable_checkpointing=False, logger=False)
    trainer.fit(model, train_loader, valid_loader)
    
    model.eval()
    trained_models[sp] = model.to(device)

# 4. Final Inference
final_predictions = []
default_model = list(trained_models.values())[0] # Fallback if species not found

with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Inference"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            img_tensor = biomass_tfs(img).unsqueeze(0).to(device)
            
            tab_tensor = torch.tensor([[
                float(safe_encode(STATE_LE, row["State"])),
                float(safe_encode(SPECIES_LE, row["Species"])),
                float(row["Pre_GSHH_NDVI"]),
                float(row["Height_Ave_cm"]),
                float(safe_encode(TARGET_NAME_LE, row["target_name"]))
            ]], dtype=torch.float32).to(device)
            
            # Use the specific model for this species
            current_sp = row["Species"]
            active_model = trained_models.get(current_sp, default_model)
            
            pred = active_model(img_tensor, tab_tensor).item()
            final_predictions.append(max(0.0, pred))
        except:
            final_predictions.append(0.0)

# 5. Create Submission (Ensuring strict Kaggle format)
submission_df = pd.DataFrame({
    "sample_id": test_pd_original["sample_id"],
    "target": final_predictions
})

submission_df.to_csv('submission.csv', index=False)
print(f"\n✅ Submission saved! Total rows: {len(submission_df)}")


=== STEP 5: Final Biomass Prediction (Multi-Model Approach) ===

--- Training Model for Species: Ryegrass_Clover ---


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone │ ResNet     │ 23.5 M │ train │     0 │
│ 1 │ tab_net  │ Sequential │  4.9 K │ train │     0 │
│ 2 │ head     │ Sequential │  266 K │ train │     0 │
│ 3 │ loss_fn  │ MSELoss    │      0 │ train │     0 │
└───┴──────────┴────────────┴────────┴───────┴───────┘

Trainable params: 23.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 23.8 M                                                                                               
Total estimated model params size (MB): 95                                                                         
Modules in train mode: 161                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Trainable params: 23.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 23.8 M                                                                                               
Total estimated model params size (MB): 95                                                                         
Modules in train mode: 161                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=5` reached.


Inference: 100%|██████████| 5/5 [00:00<00:00, 16.35it/s]


✅ Submission saved! Total rows: 5
